# 1. Analysis Goal

본 노트북의 목적은 재정의된 학습 퍼널 구조를 기반으로,
각 단계에서의 전환 패턴과 이탈 요인을 분석하는 것이다.

특히 본 분석에서는 단순한 전환율 비교에 그치지 않고,
각 단계에서의 행동 변수와 전환 간의 관계를 통계적으로 검정하고,
이를 바탕으로 실제로 의미 있는 행동 기준(threshold)을 도출하는 것을 목표로 한다.

분석은 다음과 같은 절차로 진행된다:

1. 퍼널 전환율을 통해 전체 구조를 파악
2. 각 단계별로 집단 간 차이를 통계적으로 검정
3. 효과크기를 통해 실질적인 차이의 크기를 평가
4. 유의한 변수에 대해 threshold를 탐색 (PR 기반)
5. 해당 threshold가 실제로 구분력을 가지는지 재검증

이를 통해 각 퍼널 단계에서 학습자의 이탈이 발생하는 지점을 보다 구체적으로 이해하고,
운영 관점에서 개입 가능한 기준을 도출하고자 한다.

# 2. Load Preprocessed Data

본 분석에서는 전처리가 완료된 데이터를 불러와 사용한다.

전처리 과정에서는 결측값 처리, 이상값 제거, 퍼널 정합성 확보, 
그리고 파생 변수 생성 등이 수행되었으며, 해당 과정은 별도의 전처리 노트북에서 정리되어 있다.

본 노트북에서는 이러한 전처리 결과를 바탕으로,
퍼널 구조와 행동 변수 간의 관계를 분석하는 데 집중한다.

In [ ]:
from scipy.stats import mannwhitneyu
import numpy as np
import pandas as pd

# 데이터 로드
df = pd.read_csv("course_final.csv")

# 기본 확인
df.shape

(533993, 31)

In [10]:
df.columns

Index(['course_id', 'userid_DI', 'registered', 'viewed', 'explored',
       'certified', 'final_cc_cname_DI', 'LoE_DI', 'YoB', 'gender', 'grade',
       'start_time_DI', 'last_event_DI', 'nevents', 'ndays_act', 'nplay_video',
       'nchapters', 'nforum_posts', 'viewed_missing_flag', 'duration',
       'fast_completion_flag', 'start_year', 'age_raw', 'age_missing_original',
       'age_invalid', 'age_cleaned', 'age_final', 'age', 'age_group',
       'exam_flag', 'LoE_num'],
      dtype='str')

# 2.1 Define Satisfied

본 분석에서는 기존 퍼널 구조에서 마지막 단계를 단순한 인증 여부로 정의하지 않고,
학습 행동을 기반으로 한 새로운 단계인 satisfied를 도입하였다.

이를 위해, 먼저 explored == 1이면서 certified == 1인 학습자를 기준으로
정상적인 학습 과정을 거친 집단을 정의하였다.

이 집단은 실제로 강의를 충분히 수강하고 인증 기준을 충족한 학습자들로,
각 강의에서 요구되는 학습량을 반영한다고 볼 수 있다.

이후 각 강의별 nchapters 분포에서 75th percentile을 계산하고,
해당 기준 이상으로 학습한 경우를 satisfied 상태로 정의하였다.

또한, 학습 경로의 일관성을 유지하기 위해,
explored 단계를 거치지 않은 경우는 satisfied에서 제외하였다.
이는 단순히 인증만을 목표로 한 비정상적 학습 패턴을 배제하고,
실제 학습 행동을 기반으로 한 집단을 정의하기 위함이다.

In [11]:
# 1. 정상 학습 기반 집단 정의
certified_clean = df[
    (df["certified"] == 1) &
    (df["explored"] == 1)
]

# 2. 강의별 nchapters 75% 지점 계산
p75_by_course = (
    certified_clean
    .groupby("course_id")["nchapters"]
    .quantile(0.75)
    .reset_index()
    .rename(columns={"nchapters": "nchapters_p75"})
)

# 3. merge
df = df.merge(p75_by_course, on="course_id", how="left")

# 4. satisfied 생성
df["satisfied"] = (
    (df["nchapters"] >= df["nchapters_p75"]) &
    (df["explored"] == 1)
).astype(int)

In [ ]:
# 체크
df[(df["satisfied"] == 1) & (df["explored"] == 0)]

,course_id,userid_DI,registered,viewed,explored,certified,final_cc_cname_DI,LoE_DI,YoB,gender,...,age_missing_original,age_invalid,age_cleaned,age_final,age,age_group,exam_flag,LoE_num,nchapters_p75,satisfied


# 3. Funnel Sanity Check

본 단계에서는 재정의된 퍼널 구조가 데이터 상에서 정상적으로 구성되어 있는지를 확인한다.

구체적으로는 각 단계별 학습자 수와 비율을 점검하고,
단계 간 논리적 일관성이 유지되는지를 확인한다.

이는 이후 전환율 분석 및 단계별 비교 분석을 수행하기에 앞서,
데이터가 분석 가능한 상태인지 검증하기 위한 과정이다.

In [13]:
# 단계별 인원 수
print("Registered:", df["registered"].sum())
print("Viewed:", df["viewed"].sum())
print("Explored:", df["explored"].sum())
print("Satisfied:", df["satisfied"].sum())

Registered: 533993
Viewed: 325361
Explored: 37584
Satisfied: 11870


In [14]:
# viewed 없이 explored 존재 여부
invalid_explore = df[(df["explored"] == 1) & (df["viewed"] != 1)]
print("Explored without Viewed:", len(invalid_explore))

# explored 없이 satisfied 존재 여부
invalid_satisfied = df[(df["satisfied"] == 1) & (df["explored"] != 1)]
print("Satisfied without Explored:", len(invalid_satisfied))

Explored without Viewed: 0
Satisfied without Explored: 0


### Observation

각 퍼널 단계는 전반적으로 논리적 순서를 따르고 있으며,
단계 간 역전되는 관측치는 거의 존재하지 않거나 제거된 상태임을 확인할 수 있다.

따라서 현재 데이터는 퍼널 기반 분석을 수행하기에 적절한 상태로 판단된다.

# 4. Funnel Conversion Rates

본 단계에서는 재정의된 퍼널 구조를 기반으로,
각 단계 간 전환율을 계산하여 학습자의 전체 흐름을 파악한다.

퍼널 전환율은 학습자가 다음 단계로 이동하는 비율을 의미하며,
이를 통해 학습 과정에서 이탈이 집중적으로 발생하는 구간을 식별할 수 있다.

이 분석은 이후 단계별 상세 분석을 위한 기준점으로 활용된다.

In [18]:
funnel_counts = {
    "Registered": df["registered"].sum(),
    "Viewed": df["viewed"].sum(),
    "Explored": df["explored"].sum(),
    "Satisfied": df["satisfied"].sum()
}

display(funnel_counts)

{'Registered': np.int64(533993),
 'Viewed': np.int64(325361),
 'Explored': np.int64(37584),
 'Satisfied': np.int64(11870)}

In [19]:
view_rate = df["viewed"].mean()

explore_rate = df[df["viewed"] == 1]["explored"].mean()

satisfaction_rate = df[df["explored"] == 1]["satisfied"].mean()

view_rate, explore_rate, satisfaction_rate

(np.float64(0.6092982492279861),
 np.float64(0.11551476667455535),
 np.float64(0.315825883354619))

In [21]:
funnel_df = pd.DataFrame({
    "Stage": ["Registered → Viewed", "Viewed → Explored", "Explored → Satisfied"],
    "Conversion Rate": [view_rate, explore_rate, satisfaction_rate]
})

display(funnel_df)

,Stage,Conversion Rate
0,Registered → Viewed,0.609298
1,Viewed → Explored,0.115515
2,Explored → Satisfied,0.315826


### Observation

퍼널 전환율을 살펴본 결과, Viewed → Explored 구간에서
전환율이 약 11.6%로 가장 낮게 나타났다.

이는 학습자가 강의에 진입한 이후 실제 학습으로 이어지지 않는 경우가
상당히 많음을 의미하며, 해당 구간이 주요 이탈 지점으로 작용하고 있음을 시사한다.

반면, Explored 이후 구간에서는 상대적으로 높은 전환율이 나타나,
일정 수준 이상의 학습을 시작한 경우에는 지속적인 참여로 이어질 가능성이 높음을 확인할 수 있다.

이에 따라 이후 분석에서는 각 단계 중 특히 Viewed → Explored 구간을 중심으로,
행동 변수와 전환 간의 관계를 보다 집중적으로 살펴보고자 한다.

# 5. Early-stage Analysis (Registered → Viewed)

본 단계에서는 registered에서 viewed로 이어지는 초기 구간을 분석한다.

이 구간은 학습자가 강의에 등록한 이후 실제로 강의 콘텐츠에 진입하는 단계로,
학습 과정에서의 첫 번째 전환 지점에 해당한다.

그러나 초기 단계에서는 학습 활동이 본격적으로 발생하기 이전 상태이기 때문에,
활동 기반 변수로 전환을 설명하는 것이 가능한지에 대한 검토가 선행되어야 한다.

이에 따라 본 단계에서는 먼저 활동 변수의 분포와 데이터 상태를 확인하고,
해당 변수들이 분석에 활용 가능한지를 판단한다.

## 5.1 Group Definition

초기 단계 분석을 위해 전체 학습자를 viewed 여부에 따라 두 집단으로 구분하였다.

- viewed == 0 : 강의에 진입하지 않은 집단
- viewed == 1 : 강의에 진입한 집단

이러한 구분을 통해, 초기 단계에서의 전환 여부에 따라
활동 변수의 차이가 존재하는지를 확인하고자 한다.

In [25]:
early = df.copy()

g0 = early[early["viewed"] == 0]
g1 = early[early["viewed"] == 1]

print("Viewed=0:", len(g0))
print("Viewed=1:", len(g1))

Viewed=0: 208632
Viewed=1: 325361


In [27]:
cols = ["nevents", "ndays_act", "nplay_video", "nforum_posts"]

for col in cols:
    x = g0[col].dropna()
    y = g1[col].dropna()

    if len(x) == 0 or len(y) == 0:
        continue

    stat, p = mannwhitneyu(x, y, alternative="two-sided")
    delta = (2 * stat) / (len(x) * len(y)) - 1

    print(f"\n==== {col} ====")
    print("p-value:", p)
    print("Cliff's delta:", delta)
    print("median g0:", np.median(x))
    print("median g1:", np.median(y))


==== nevents ====
p-value: 0.0
Cliff's delta: -0.9503709144487793
median g0: 1.0
median g1: 64.0

==== ndays_act ====
p-value: 0.0
Cliff's delta: -0.7792409843892918
median g0: 1.0
median g1: 3.0

==== nplay_video ====
p-value: 0.0
Cliff's delta: -0.9997940788771295
median g0: 0.0
median g1: 18.0

==== nforum_posts ====
p-value: 0.0
Cliff's delta: -0.022395681987147253
median g0: 0.0
median g1: 0.0


## 5.2 Data Availability Check

초기 단계에서는 활동 변수(nevents, ndays_act, nplay_video, nforum_posts)가
viewed 여부를 설명하는 데 활용 가능한지를 확인하였다.

그러나 해당 변수들은 대부분의 관측치에서 값이 매우 낮거나,
결측 비율이 높은 특성을 보였다.

이는 registered → viewed 구간에서는 학습 활동 자체가 거의 발생하지 않는 초기 단계이기 때문에,
활동 기반 지표가 아직 충분히 축적되지 않은 상태임을 의미한다.

따라서 이 단계에서는 활동 변수 간의 통계적 차이를 검정하더라도,
그 결과를 해석하는 데에는 한계가 존재한다고 판단하였다.

In [26]:
cols = ["nevents", "ndays_act", "nplay_video", "nforum_posts"]

for col in cols:
    print(f"\n==== {col} ====")
    print("median:", df[col].median())
    print("mean:", df[col].mean())
    print("non-zero ratio:", (df[col] > 0).mean())
    print("missing ratio:", df[col].isna().mean())


==== nevents ====
median: 11.0
mean: 357.68204161702863
non-zero ratio: 0.8251793562836967
missing ratio: 0.0033839394898435

==== ndays_act ====
median: 1.0
mean: 4.8251118969683535
non-zero ratio: 0.8251793562836967
missing ratio: 0.0033839394898435

==== nplay_video ====
median: 1.0
mean: 60.31353998888004
non-zero ratio: 0.3396767373355081
missing ratio: 0.3465794495433461

==== nforum_posts ====
median: 0.0
mean: 0.022651982329356377
non-zero ratio: 0.013865350294854053
missing ratio: 0.0


In [ ]:
for col in cols:
    print(f"\n==== {col} ====")
    print("Viewed=0 median:", g0[col].median())
    print("Viewed=1 median:", g1[col].median())